# 02 · Agent execution loop with error feedback

The smallest self-correcting coding agent: Claude (on **Amazon Bedrock**,
Converse API) writes Python, the sandbox runs it, and any traceback is fed back
so the model fixes its own mistake.

**Concepts:** the sandbox is ground truth — you *run* generated code instead of
trusting it; the returned error drives a bounded retry loop. Real model, no stubs.

**Prerequisites:** AWS credentials + Bedrock model access for `RAITBOX_BEDROCK_MODEL`.

In [ ]:
%pip install -q raitbox  # boto3 comes with it

### Configuration & model call
Region + Bedrock model, the `bedrock-runtime` client, and `ask_model` (Converse, fences stripped).

In [ ]:
import os
import re

import boto3

from raitbox import Sandbox

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")
MODEL_ID = os.environ.get("RAITBOX_BEDROCK_MODEL", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")
bedrock = boto3.client("bedrock-runtime", region_name=REGION)

SYSTEM = [{"text": "You write a short Python program for the task. Reply with ONLY code "
           "(no prose, no fences). Given a traceback from a previous attempt, fix it."}]


def ask_model(messages) -> str:
    resp = bedrock.converse(modelId=MODEL_ID, system=SYSTEM, messages=messages,
                            inferenceConfig={"maxTokens": 1024, "temperature": 0})
    text = "".join(b.get("text", "") for b in resp["output"]["message"]["content"])
    fenced = re.findall(r"```(?:python)?\n(.*?)```", text, re.DOTALL)
    return (fenced[0] if fenced else text).strip()

### The feedback loop
Run the code; on error, append the traceback and ask again, up to a budget.

In [ ]:
def run_agent(task: str, max_attempts: int = 4) -> str | None:
    messages = [{"role": "user", "content": [{"text": task}]}]
    with Sandbox.create(region_name=REGION) as sbx:
        for attempt in range(1, max_attempts + 1):
            code = ask_model(messages)
            print(f"--- attempt {attempt} ---\n{code}\n")
            execution = sbx.run_code(code)
            if execution.error is None:
                return "".join(execution.logs.stdout)
            err = f"{execution.error.name}: {execution.error.value}"
            print("error:", err)
            messages.append({"role": "assistant", "content": [{"text": code}]})
            messages.append({"role": "user", "content": [{"text": f"That failed:\n{err}\nFix it."}]})
    return None

### Run it

In [ ]:
task = ("Among all starting numbers below 1000, print the one whose Collatz "
        "sequence is the longest, together with that length.")
print("result:", run_agent(task))

### Recap & next
Add a *tool* (`03-claude-code-interpreter`) or iterate against a real test suite (`06-iterative-self-repair`).